In [5]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, RobustScaler, OneHotEncoder
from sklearn.preprocessing import FunctionTransformer

from sklearn.feature_selection import SelectFromModel
from sklearn.ensemble import RandomForestClassifier, IsolationForest

In [6]:
data_path = "train_processed_advanced.csv"

df = pd.read_csv(data_path)

print("Dataset Shape:", df.shape)

df.head()

Dataset Shape: (336715, 54)


,duration,src_bytes,dst_bytes,land,wrong_fragment,urgent,hot,num_failed_logins,logged_in,num_compromised,...,flag_REJ,flag_RSTO,flag_RSTOS0,flag_RSTR,flag_S1,flag_S2,flag_S3,flag_SF,flag_SH,attack_class
0,0.0,1.619565,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,normal
1,0.0,0.369565,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,normal
2,0.0,-0.159420,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-1.0,0.0,DoS
3,0.0,0.681159,15.800388,0.0,0.0,0.0,0.0,0.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,normal
4,0.0,0.561594,0.813953,0.0,0.0,0.0,0.0,0.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,normal


In [7]:
target_column = df.columns[-1]

print("Target column:", target_column)

Target column: attack_class


In [8]:
X = df.drop(target_column, axis=1)

y = df[target_column]

print("Feature shape:", X.shape)
print("Target shape:", y.shape)

Feature shape: (336715, 53)
Target shape: (336715,)


In [9]:
num_cols = X.select_dtypes(include=['int64','float64']).columns
cat_cols = X.select_dtypes(include=['object']).columns

print("Numerical Columns:", len(num_cols))
print("Categorical Columns:", len(cat_cols))

Numerical Columns: 53
Categorical Columns: 0


In [10]:
X[num_cols] = X[num_cols].bfill()

X[num_cols] = X[num_cols].ffill()

In [18]:
from sklearn.preprocessing import FunctionTransformer, RobustScaler
from sklearn.pipeline import Pipeline
import numpy as np

# Safe log transform function
def safe_log_transform(X):
    X = np.clip(X, a_min=0, a_max=None)  # remove negative values
    return np.log1p(X)

signal_pipeline = Pipeline([

    ("safe_log", FunctionTransformer(safe_log_transform)),

    ("robust_scaler", RobustScaler())

])

In [19]:
num_pipeline = Pipeline([

    ("imputer", SimpleImputer(strategy="median")),

    ("scaler", StandardScaler())

])

In [20]:
preprocessor = ColumnTransformer([

    ("num", num_pipeline, num_cols)

])

In [21]:
feature_selector = SelectFromModel(

    RandomForestClassifier(

        n_estimators=150,
        random_state=42,
        n_jobs=-1

    )

)

In [22]:
model_pipeline = Pipeline([

    ("preprocessing", preprocessor),

    ("signal_transform", signal_pipeline),

    ("feature_selection", feature_selector),

    ("classifier",

        RandomForestClassifier(

            n_estimators=250,
            random_state=42,
            n_jobs=-1

        )

    )

])

In [23]:
X_train, X_test, y_train, y_test = train_test_split(

    X,
    y,
    test_size=0.2,
    random_state=42

)

In [24]:
model_pipeline.fit(X_train, y_train)

Pipeline(steps=[('preprocessing',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  Index(['duration', 'src_bytes', 'dst_bytes', 'land', 'wrong_fragment',
       'urgent', 'hot', 'num_failed_logins', 'logged_in', 'num_compromised',
       'root_shell', 'su_attempted', 'num_file_creations', 'num_sh...
                ('signal_transform',
                 Pipeline(steps=[('safe_log',
                                  FunctionTransformer(func=<function safe_log_transform at 0x00000210E2E918A0>)),
                                 ('robust_scaler', RobustScaler())])),
                ('feature_selection',
                 SelectFromModel(estimator=RandomForestClassifier(n_estimators=150,
                                                                  n_jobs=-1,
                                                                  random_state=42))),
                ('classifier',
                 RandomForestClassifier(n_estimators=250, n_jobs=-1,
                                        random_state=42))])

In [25]:
def safe_log_transform(X):
    X = np.clip(X, a_min=0, a_max=None)
    return np.log1p(X)

signal_pipeline = Pipeline([
    ("log_transform", FunctionTransformer(safe_log_transform)),
    ("robust_scaler", RobustScaler())
])

In [26]:
X_signal = signal_pipeline.fit_transform(X[num_cols])

print("Signal transformation completed")

print(X_signal.shape)

Signal transformation completed
(336715, 53)


In [27]:
from sklearn.ensemble import IsolationForest

iso_model = IsolationForest(

    n_estimators=200,
    contamination=0.05,
    random_state=42

)

iso_model.fit(X_signal)

anomaly_pred = iso_model.predict(X_signal)